# Coding Exercise: Decoding a Secret Message

Given a published Google Doc containing a table of `(x, character, y)` triples, print the grid of characters they describe. When rendered in a fixed-width font, the grid reveals a secret message made of uppercase letters.

In [ ]:
from dataclasses import dataclass

import requests
from bs4 import BeautifulSoup


class GridData:
    characters: dict  # (x, y) -> character
    width: int
    height: int


def _fetch_table_rows(doc_url: str) -> list:
    html = requests.get(doc_url).text
    table = BeautifulSoup(html, "html.parser").find("table")
    if table is None:
        raise ValueError("No table found in document")
    return table.find_all("tr")


def _locate_columns(header_row) -> dict:
    headers = [cell.get_text(strip=True).lower() for cell in header_row.find_all(["td", "th"])]
    return {
        field: next(i for i, header in enumerate(headers) if field in header)
        for field in ("x", "y", "char")
    }


def _parse_grid_data(doc_url: str) -> GridData:
    rows = _fetch_table_rows(doc_url)
    columns = _locate_columns(rows[0])

    characters = {}
    width = height = 0

    for row in rows[1:]:
        cells = row.find_all(["td", "th"])
        if len(cells) < 3:
            continue

        x = int(cells[columns["x"]].get_text(strip=True))
        y = int(cells[columns["y"]].get_text(strip=True))
        char = cells[columns["char"]].get_text(strip=True)

        characters[(x, y)] = char
        width = max(width, x + 1)
        height = max(height, y + 1)

    return GridData(characters, width, height)


def _render_grid(grid: GridData) -> str:
    return "\n".join(
        "".join(grid.characters.get((x, y), " ") for x in range(grid.width))
        for y in range(grid.height - 1, -1, -1)
    )


def print_secret_message(doc_url: str) -> None:
    print(_render_grid(_parse_grid_data(doc_url)))

In [7]:
print_secret_message("https://docs.google.com/document/d/e/2PACX-1vTMOmshQe8YvaRXi6gEPKKlsC6UpFJSMAk4mQjLm_u1gmHdVVTaeh7nBNFBRlui0sTZ-snGwZM4DBCT/pub")

█▀▀▀
█▀▀ 
█   


In [16]:
print_secret_message("https://docs.google.com/document/d/e/2PACX-1vSvM5gDlNvt7npYHhp_XfsJvuntUhq184By5xO_pA4b_gCWeXb6dM6ZxwN8rE6S4ghUsCj2VKR21oEP/pub")


██░     ██░    ███████░     ██░     ██░     ██████░ ████████░    ████████░     ████████░  
██░     ██░  ███░    ██░   ████░   ████░      ██░   ██░     ██░  ██░     ██░ ███░     ███░
██░     ██░ ███░           ██░██░ ██░██░      ██░   ██░      ██░ ██░     ██░ ██░       ██░
██████████░ ██░           ███░ ██░██░ ██░     ██░   ██░      ██░ ████████░   ██░       ██░
██░     ██░ ███░          ██░  █████░ ███░    ██░   ██░      ██░ ██░     ██░ ██░       ██░
██░     ██░  ███░    ██░ ███░   ███░   ██░    ██░   ██░     ██░  ██░     ██░ ███░     ███░
██░     ██░    ███████░  ██░           ███░ ██████░ ████████░    ████████░     ████████░  
